# 02 — Emotion word list and prompts

**Goal of this notebook.** Lock in *what* we will generate before *how*. Three things land here:

1. A **generator model** to write the emotion-conditioned passages — chosen with eyes open about quality, licence, and Colab fit.
2. A **171-emotion vocabulary** spanning the valence × arousal plane, plus paired **neutral baselines** (the contrast that becomes the difference-of-means direction in nb05).
3. A **wall-clock estimate** for the full generation pass on Colab T4 / L4 / A100 vs. local hardware — so we know what we're committing to before we kick it off in nb03.

We follow Anthropic (*transformer-circuits.pub/2026/emotions*) faithfully on the prompt structure: short character stories that *experience* a target emotion, paired against neutral counterparts. The notebook emits a small JSON spec that `03_story_generation.ipynb` consumes.

> **Output of this notebook:** `data/processed/dataset_spec.json` — a frozen specification of model, emotion list, prompts, and generation budget. Re-running nb02 with the same config is a no-op.

## 1 — Setup

In [ ]:
# Self-installing on Colab (skipped if already present locally).
import importlib, subprocess, sys
def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for pkg, imp in [("transformers", "transformers"), ("matplotlib", "matplotlib"), ("numpy", "numpy")]:
    _ensure(pkg, imp)


In [ ]:
import json, os, sys, time, math, hashlib, textwrap
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt

# Where to write the dataset spec. On Colab, mount Drive optionally; otherwise local.
try:
    from google.colab import drive  # noqa
    IS_COLAB = True
    WORK_DIR = Path("/content/EmoVecLLM"); WORK_DIR.mkdir(exist_ok=True)
except Exception:
    IS_COLAB = False
    WORK_DIR = Path("..").resolve()  # repo root when notebook is in notebooks/

DATA_DIR = WORK_DIR / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"IS_COLAB={IS_COLAB}  |  WORK_DIR={WORK_DIR}  |  DATA_DIR={DATA_DIR}")


## 2 — Choose a generator model

Anthropic uses **Claude Sonnet 4.5** to write the stories. We can't (it's closed), so we pick from open-weight models that fit the constraints below.

**What the generator must do.**
- Follow an instruction reliably (chat / instruct-tuned, not a base LM).
- Produce ~200–300-word coherent character vignettes per emotion.
- Be small enough to live alongside the *target* model in VRAM, **or** be run in a separate session and the stories cached to disk.

**What the generator does *not* need to do.**
- Be the same model we extract activations from — those are independent choices.
- Be huge. Anthropic's exact wording matters less than getting *recognisable emotional content* per passage; even a 7-8B instruct model produces fine raw material.

| Model | Params | T4 (free Colab) | L4 / A100 (Pro) | Licence | Notes |
|---|---|---|---|---|---|
| `gpt2` | 124 M | fp16 ✅ | fp16 ✅ | MIT | Too small for instruction-following — base only, **not viable** as generator. |
| `EleutherAI/pythia-1.4b` | 1.4 B | fp16 ✅ | fp16 ✅ | Apache-2.0 | Base model, no instruction tuning. Usable for prototyping with few-shot prompting; expect uneven story quality. |
| `Qwen/Qwen2.5-7B-Instruct` | 7 B | 4-bit ✅ | fp16 ✅ | Tongyi Qianwen | Strong instruction following; multilingual; permissive but not Apache. |
| `meta-llama/Llama-3.1-8B-Instruct` | 8 B | 4-bit ✅ | fp16 ✅ | Llama-3 (gated) | Headline open generator; needs HF token + license accept. |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7 B | 4-bit ✅ | fp16 ✅ | Apache-2.0 | Drop-in alternative when Llama gating is inconvenient. |

**Our default below.** `Qwen2.5-7B-Instruct` — Apache-permissive, strong, no gating — for ease of replication. Anyone can clone and run nb03 without HF access negotiations. Override `GENERATOR_MODEL` if you have HF credentials and prefer Llama-3.

In [ ]:
# ── Configurable generator model ──────────────────────────────────────────────
GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Generation knobs (held constant across emotions for fair contrast).
N_STORIES_PER_EMOTION = 5      # Anthropic-style multiple draws per concept
TARGET_TOKENS_PER_STORY = 256  # ≈200 words; long enough to carry an emotion
TEMPERATURE = 0.9              # creative but not chaotic
TOP_P = 0.95
SEED_BASE = 0xE3F0             # +i*hash(emotion) for per-prompt reproducibility

# Hardware profile (for runtime estimate; choose what you actually run on).
HW_PROFILE = "T4_4bit"   # one of: T4_4bit, T4_fp16, L4_fp16, A100_fp16, RTX4090, CPU

print(f"Generator: {GENERATOR_MODEL}")
print(f"Per emotion: {N_STORIES_PER_EMOTION} stories × {TARGET_TOKENS_PER_STORY} tokens")
print(f"Hardware profile for time estimate: {HW_PROFILE}")


## 3 — The 171-emotion vocabulary

Anthropic seed their pipeline with a curated list of **171 emotion concepts** spanning:

- **Valence** (pleasant ↔ unpleasant)
- **Arousal** (high activation ↔ low activation)
- **Granularity** — non-basic categories such as *brooding*, *infatuated*, *valiant*, *droopy*, *dumbstruck*. The point is **not** the six basic emotions; it is the fine-grained semantic structure the model carries internally.

The list is **published verbatim** in the paper's appendix ("Full list of emotions" section). We use it directly. It includes a few entries that read more as *traits or states* than canonical emotions (*alert*, *kind*, *patient*, *dependent*, *stuck*, *reflective*) — Anthropic appear to use a permissive *affective concept* definition rather than basic-emotion-theory criteria.

A full source note with the verbatim list, the 100 story-topic seeds, and the prompt templates lives at:

> `../../EmotionVectorsReview/canonical_emotions_and_prompts.md`

The fitted V/A regression onto our derived vectors happens in nb07 against the actual Warriner et al. 2013 norms — we don't pre-tag here.

In [ ]:
# Canonical 171-emotion list from Anthropic (April 2026), transformer-circuits.pub/2026/emotions
# §"Full list of emotions". Verbatim — see ../../EmotionVectorsReview/canonical_emotions_and_prompts.md
# for the full source note (also captures the 100 topic seeds and prompt templates).
EMOTIONS_171 = [
    'afraid', 'alarmed', 'alert', 'amazed', 'amused', 'angry',
    'annoyed', 'anxious', 'aroused', 'ashamed', 'astonished', 'at ease',
    'awestruck', 'bewildered', 'bitter', 'blissful', 'bored', 'brooding',
    'calm', 'cheerful', 'compassionate', 'contemptuous', 'content', 'defiant',
    'delighted', 'dependent', 'depressed', 'desperate', 'disdainful', 'disgusted',
    'disoriented', 'dispirited', 'distressed', 'disturbed', 'docile', 'droopy',
    'dumbstruck', 'eager', 'ecstatic', 'elated', 'embarrassed', 'empathetic',
    'energized', 'enraged', 'enthusiastic', 'envious', 'euphoric', 'exasperated',
    'excited', 'exuberant', 'frightened', 'frustrated', 'fulfilled', 'furious',
    'gloomy', 'grateful', 'greedy', 'grief-stricken', 'grumpy', 'guilty',
    'happy', 'hateful', 'heartbroken', 'hope', 'hopeful', 'horrified',
    'hostile', 'humiliated', 'hurt', 'hysterical', 'impatient', 'indifferent',
    'indignant', 'infatuated', 'inspired', 'insulted', 'invigorated', 'irate',
    'irritated', 'jealous', 'joyful', 'jubilant', 'kind', 'lazy',
    'listless', 'lonely', 'loving', 'mad', 'melancholy', 'miserable',
    'mortified', 'mystified', 'nervous', 'nostalgic', 'obstinate', 'offended',
    'on edge', 'optimistic', 'outraged', 'overwhelmed', 'panicked', 'paranoid',
    'patient', 'peaceful', 'perplexed', 'playful', 'pleased', 'proud',
    'puzzled', 'rattled', 'reflective', 'refreshed', 'regretful', 'rejuvenated',
    'relaxed', 'relieved', 'remorseful', 'resentful', 'resigned', 'restless',
    'sad', 'safe', 'satisfied', 'scared', 'scornful', 'self-confident',
    'self-conscious', 'self-critical', 'sensitive', 'sentimental', 'serene', 'shaken',
    'shocked', 'skeptical', 'sleepy', 'sluggish', 'smug', 'sorry',
    'spiteful', 'stimulated', 'stressed', 'stubborn', 'stuck', 'sullen',
    'surprised', 'suspicious', 'sympathetic', 'tense', 'terrified', 'thankful',
    'thrilled', 'tired', 'tormented', 'trapped', 'triumphant', 'troubled',
    'uneasy', 'unhappy', 'unnerved', 'unsettled', 'upset', 'valiant',
    'vengeful', 'vibrant', 'vigilant', 'vindictive', 'vulnerable', 'weary',
    'worn out', 'worried', 'worthless',
]
assert len(EMOTIONS_171) == 171
assert len(set(EMOTIONS_171)) == 171, "duplicate emotion word!"
print(f"Loaded {len(EMOTIONS_171)} emotions (Anthropic canonical list).")
print(f"first 6: {EMOTIONS_171[:6]}")
print(f"last 6:  {EMOTIONS_171[-6:]}")

# Multi-word entries: handle as full strings; slugify only at filesystem boundary.
multiword = [w for w in EMOTIONS_171 if " " in w or "-" in w]
print(f"\nmulti-word / hyphenated entries ({len(multiword)}): {multiword}")


### 3a — Survey the list

A quick visual of the 171 words by length and first letter — purely informational, to make the list legible at a glance.

In [ ]:
from collections import Counter
import string

# Word-length distribution (interesting because longer words tend to be granular / non-basic emotions)
lengths = [len(w) for w in EMOTIONS_171]
first_letters = Counter(w[0] for w in EMOTIONS_171)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), constrained_layout=True)

axes[0].hist(lengths, bins=range(2, max(lengths)+2), color="#34618d", edgecolor="white")
axes[0].set_xlabel("word length (characters)")
axes[0].set_ylabel("count")
axes[0].set_title("Length distribution")
axes[0].spines[["top", "right"]].set_visible(False)

letters_sorted = sorted(first_letters.keys())
axes[1].bar(letters_sorted, [first_letters[c] for c in letters_sorted], color="#3a7d44")
axes[1].set_xlabel("first letter")
axes[1].set_ylabel("count")
axes[1].set_title("Distribution by first letter")
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle(f"171 canonical emotions — overview", fontsize=11)
plt.show()

# Print the list grouped by first letter for readability
print()
print("─" * 70)
for c in string.ascii_lowercase:
    words = [w for w in EMOTIONS_171 if w[0] == c]
    if words:
        print(f"  {c}: {', '.join(words)}")


## 4 — Prompt templates

Two prompts per emotion: an **emotion-conditioned** story and a **neutral** baseline. Both are matched in form (length, register, character-driven, present-tense), differing only in whether the emotional concept is invoked.

This matched-pair structure is what makes the difference-of-means in nb05 a clean *contrast*: anything else that varies between the two distributions (genre, sentence length, narrator voice) gets averaged out across stories. Length-matching matters — if emotion stories are systematically longer, the residual norm in nb04 will pick up "more text" rather than "more emotion".

### Why character stories?

Anthropic's choice to ask the model to *write a character experiencing X* rather than *describe X* is deliberate. It pushes the residual stream towards the **simulation** of the emotion (an internal representation in service of generation) rather than a **dictionary lookup**. The vectors we will derive are most useful when they reflect the former.

### The templates

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful, vivid creative writer. "
    "When asked, you write short character vignettes in plain modern prose. "
    "You stay in scene, third person, present tense. No headings, no preamble, no meta-commentary."
)

EMOTION_USER_TEMPLATE = (
    "Write a short story (about 200 words) about a character who is feeling {emotion}. "
    "Show this emotion through their thoughts, actions, sensations, and what they notice — do not "
    "name the emotion outright. Begin in the middle of a moment. Keep it concrete."
)

NEUTRAL_USER_TEMPLATE = (
    "Write a short story (about 200 words) about a character going about an ordinary moment of their day "
    "(e.g., walking to a shop, reading a letter, looking out a window). Stay neutral in tone — neither "
    "uplifted nor distressed. Show this through their thoughts, actions, sensations, and what they notice. "
    "Begin in the middle of a moment. Keep it concrete."
)

# Demo on three example emotions
for w in ["desperate", "calm", "broody"]:
    print(f"── emotion: {w} ──")
    print(EMOTION_USER_TEMPLATE.format(emotion=w))
    print()
print("── neutral baseline ──")
print(NEUTRAL_USER_TEMPLATE)


### 4a — Sanity-check tokenisation

Different generators tokenise differently — a 200-word target maps to very different token counts on GPT-2 vs. Llama vs. Qwen. We check the prompt length and a placeholder generated story to refine the token budget.

We don't load the full generator (heavy). The tokenizer alone is ~10 MB and is enough to estimate.

In [ ]:
from transformers import AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained(GENERATOR_MODEL, trust_remote_code=True)
    USING_REAL_TOK = True
except Exception as e:
    print(f"(falling back to gpt2 tokenizer for sizing: {e})")
    tok = AutoTokenizer.from_pretrained("gpt2")
    USING_REAL_TOK = False

# Build the chat-formatted prompt the way the generator will see it.
def chat_prompt(user_msg: str) -> str:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg}]
    if hasattr(tok, "apply_chat_template") and USING_REAL_TOK:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return f"<|sys|>{SYSTEM_PROMPT}\n<|user|>{user_msg}\n<|assistant|>"

example_emotion_prompt = chat_prompt(EMOTION_USER_TEMPLATE.format(emotion="desperate"))
example_neutral_prompt = chat_prompt(NEUTRAL_USER_TEMPLATE)

# Token lengths
n_tok_emotion_prompt = len(tok(example_emotion_prompt).input_ids)
n_tok_neutral_prompt = len(tok(example_neutral_prompt).input_ids)
print(f"tokenizer: {GENERATOR_MODEL if USING_REAL_TOK else 'gpt2 (fallback)'}")
print(f"emotion-prompt tokens: {n_tok_emotion_prompt}")
print(f"neutral-prompt tokens: {n_tok_neutral_prompt}")
print(f"target output tokens : {TARGET_TOKENS_PER_STORY}")
print(f"per-call wall budget : {n_tok_emotion_prompt + TARGET_TOKENS_PER_STORY} tokens (prompt + output)")


## 5 — How long will this take?

The total cost is dominated by **decode** — the autoregressive generation pass — not the (tiny) prompt prefill. Decode throughput depends almost entirely on (model size × precision × hardware), so we can table it.

Numbers below are realistic ballpark figures for **Llama-3.1-8B-class** instruct models. They come from published `llama.cpp` / `vLLM` / `transformers` benchmarks and our own measurements; expect ±30 % variance per run. Use these to *plan*, not to predict to the second.

In [ ]:
# Throughput table: tokens/sec of *decode* for an 8B-class instruct model.
THROUGHPUT_TOK_S = {
    # profile           : tok/s
    "T4_4bit":   28,    # bitsandbytes 4-bit, batch 1 — the free-Colab baseline
    "T4_fp16":   12,    # 8B fp16 barely fits → KV-cache fragmentation hurts
    "L4_fp16":   75,    # Colab Pro mid-tier
    "A100_fp16": 130,   # Colab Pro A100 — best on-platform
    "RTX4090":   105,   # local consumer card, fp16
    "CPU":       2.5,   # local CPU-only, llama.cpp Q4 — included for the cautionary tale
}

# For smaller generators the throughput scales roughly inversely with params.
PARAM_SCALE = {  # approximate vs 8B-class
    "Qwen/Qwen2.5-7B-Instruct":          1.10,
    "meta-llama/Llama-3.1-8B-Instruct":  1.00,
    "mistralai/Mistral-7B-Instruct-v0.3":1.10,
    "EleutherAI/pythia-1.4b":            5.00,   # 5× faster than 8B
    "gpt2":                              35.0,
}
SCALE = PARAM_SCALE.get(GENERATOR_MODEL, 1.0)
EFFECTIVE_THROUGHPUT = THROUGHPUT_TOK_S[HW_PROFILE] * SCALE
print(f"profile {HW_PROFILE} on 8B-class: {THROUGHPUT_TOK_S[HW_PROFILE]} tok/s")
print(f"× {SCALE:.2f} for {GENERATOR_MODEL} → {EFFECTIVE_THROUGHPUT:.0f} tok/s effective decode")


In [ ]:
# Generation budget
N_EMOTIONS = 171
N_TOTAL_STORIES_EMOTION = N_EMOTIONS * N_STORIES_PER_EMOTION
N_TOTAL_STORIES_NEUTRAL = N_TOTAL_STORIES_EMOTION  # paired baseline of equal volume
N_TOTAL_STORIES = N_TOTAL_STORIES_EMOTION + N_TOTAL_STORIES_NEUTRAL

decode_tokens = N_TOTAL_STORIES * TARGET_TOKENS_PER_STORY
prefill_tokens = N_TOTAL_STORIES * (n_tok_emotion_prompt + n_tok_neutral_prompt) // 2

# Wall-clock: dominated by decode (prefill is a one-shot batched pass per call).
secs_decode = decode_tokens / EFFECTIVE_THROUGHPUT
# Add ~5 % overhead for prefill + sampling + scheduling.
secs_total = secs_decode * 1.05

def fmt(s):
    h = int(s // 3600); m = int((s % 3600) // 60); ss = int(s % 60)
    if h: return f"{h}h {m:02d}m {ss:02d}s"
    if m: return f"{m}m {ss:02d}s"
    return f"{ss}s"

print(f"emotion stories: {N_TOTAL_STORIES_EMOTION}")
print(f"neutral stories: {N_TOTAL_STORIES_NEUTRAL}")
print(f"total stories  : {N_TOTAL_STORIES}")
print(f"decode tokens  : {decode_tokens:,}")
print(f"prefill tokens : {prefill_tokens:,} (negligible cost vs decode)")
print()
print(f"≈ wall-clock on {HW_PROFILE} for {GENERATOR_MODEL}: {fmt(secs_total)}")


### 5a — Compare across hardware

The same 171-emotion budget swept across every hardware profile and every viable generator. Read this as a *budget table*, not a *recommendation* — even a slow run is fine if you only do it once and cache the output.

In [ ]:
rows = []
for prof, base_tok_s in THROUGHPUT_TOK_S.items():
    for model, scale in PARAM_SCALE.items():
        eff = base_tok_s * scale
        secs = (decode_tokens / eff) * 1.05
        rows.append((prof, model, eff, secs))

# Pretty-print as table
print(f"{'profile':<12} {'model':<40} {'tok/s':>7} {'wall-clock':>13}")
print("-" * 76)
for prof, model, eff, secs in rows:
    short = model.split("/")[-1]
    print(f"{prof:<12} {short:<40} {eff:>7.0f} {fmt(secs):>13}")


In [ ]:
# Visualise the same table as a heatmap (log-scaled wall-clock).
import numpy as np
profiles = list(THROUGHPUT_TOK_S.keys())
models = list(PARAM_SCALE.keys())
short_models = [m.split("/")[-1] for m in models]

mat = np.zeros((len(profiles), len(models)))
for i, p in enumerate(profiles):
    for j, m in enumerate(models):
        eff = THROUGHPUT_TOK_S[p] * PARAM_SCALE[m]
        mat[i, j] = (decode_tokens / eff) * 1.05  # seconds

fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
im = ax.imshow(np.log10(mat / 60.0), cmap="viridis", aspect="auto")  # log10(minutes)
ax.set_xticks(range(len(models))); ax.set_xticklabels(short_models, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(profiles))); ax.set_yticklabels(profiles, fontsize=9)
for i in range(len(profiles)):
    for j in range(len(models)):
        ax.text(j, i, fmt(mat[i, j]), ha="center", va="center",
                fontsize=7, color="white" if np.log10(mat[i, j]/60) > 1 else "black")
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label("log₁₀(minutes)")
ax.set_title(f"Full {N_TOTAL_STORIES}-story budget — wall-clock per (hardware × generator)", fontsize=10)
plt.show()


**Reading the heatmap.** Each cell shows the total wall-clock to generate the full paired dataset (171 × 5 emotion + 171 × 5 neutral = 1,710 stories of 256 tokens each).

- The free-Colab T4 with 8B-class 4-bit is in the **2-3 hour band**. Workable as a single overnight notebook; not workable as something you iterate on.
- Pythia-1.4B on T4 drops to ~20 minutes — fine for a prototype, but expect rougher prose without instruction tuning.
- A100 fp16 with 8B is the comfortable "I can iterate" zone (~25 min per full pass).
- CPU-only is included as a cautionary baseline — don't.

**Practical takeaway.** Generate once on whatever you have, **cache to disk**, and never regenerate unless the prompt template changes. nb03 will hash the spec we save below and short-circuit if nothing has changed.

## 6 — Freeze the dataset spec

Write a single JSON file that nb03 reads. Anything that affects what gets generated lives in this spec; the spec hash becomes the cache key.

In [ ]:
spec = {
    "version": 1,
    "generator_model": GENERATOR_MODEL,
    "decoding": {
        "n_stories_per_emotion": N_STORIES_PER_EMOTION,
        "target_tokens_per_story": TARGET_TOKENS_PER_STORY,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "seed_base": SEED_BASE,
    },
    "emotions": list(EMOTIONS_171),
    "prompts": {
        "system": SYSTEM_PROMPT,
        "emotion_template": EMOTION_USER_TEMPLATE,
        "neutral_template": NEUTRAL_USER_TEMPLATE,
    },
    "budget": {
        "n_emotion_stories": N_TOTAL_STORIES_EMOTION,
        "n_neutral_stories": N_TOTAL_STORIES_NEUTRAL,
        "n_total_stories":   N_TOTAL_STORIES,
        "decode_tokens":     decode_tokens,
    },
}

# Hash the spec for cache-keying in nb03.
spec_blob = json.dumps(spec, sort_keys=True, ensure_ascii=False).encode("utf-8")
spec_hash = hashlib.sha256(spec_blob).hexdigest()[:12]
spec["spec_hash"] = spec_hash

out_path = DATA_DIR / "dataset_spec.json"
out_path.write_text(json.dumps(spec, indent=2, ensure_ascii=False))
print(f"wrote spec → {out_path}")
print(f"spec_hash = {spec_hash}")
print(f"size      = {out_path.stat().st_size:,} bytes")


## 7 — Next

`03_story_generation.ipynb` will:

1. Load `dataset_spec.json`.
2. Load `GENERATOR_MODEL` (with sensible 4-bit fallback on small GPUs).
3. Generate `N_TOTAL_STORIES` passages, batched, with the seed schedule defined here.
4. Write outputs to `data/processed/stories/{spec_hash}/{emotion}/{i}.txt` (and a parallel neutral set).
5. Keep a manifest with per-story metadata (seed, n_tokens, finish_reason, wall-clock).

If anything in this notebook's spec changes (model, prompt, budget), `spec_hash` shifts and nb03 will treat it as a fresh run.

**Open follow-ups for this stage** (tracked in `meta/TODO.md`):

- ✅ Canonical 171-emotion list now in place from Anthropic's appendix (was: hand-curated placeholder).
- Qualitative spot-check: pick 5 emotions, generate one story each end-to-end, confirm the output reads as the intended emotion to a human reader (and **doesn't** name the emotion).
- Decide pooling rule for nb04: full passage vs. emotion-evoking sentence span. Log decision in `meta/DECISIONS.md`.